In [6]:
"""
DEMO: JsonOutputParser + Runnable methods (LangChain)
Author: Exam-oriented reference
"""

from typing import Dict, Any, List
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda


# -----------------------------
# 1️⃣ Define Output Schema
# -----------------------------
class MovieInfo(BaseModel):
    """Schema used to validate parsed LLM output"""
    title: str = Field(..., description="Movie title")
    year: int = Field(..., description="Release year")
    director: str = Field(..., description="Director name")


# -----------------------------
# 2️⃣ Create JsonOutputParser
# -----------------------------
parser = JsonOutputParser(pydantic_object=MovieInfo)

# -----------------------------
# 3️⃣ Dummy Runnable (mock LLM)
# -----------------------------
def fake_llm(prompt: str) -> Dict[str, Any]:
    """Simulates LLM JSON output"""
    return {
        "title": "Iron Man",
        "year": 2008,
        "director": "Jon Favreau"
    }

llm_runnable = RunnableLambda(fake_llm)

# -----------------------------
# 4️⃣ Pipe (|) Runnable
# -----------------------------
chain = llm_runnable | parser  # Pipes LLM output into parser

In [7]:
# =============================
# 🔹 Runnable Metadata Methods
# =============================

print(chain.get_name())  # Returns runnable name
print(chain.get_input_schema())  # Pydantic model for input validation
print(chain.get_input_jsonschema())  # JSON schema of input
print(chain.get_output_schema())  # Pydantic output schema
print(chain.get_output_jsonschema())  # JSON schema of output
print(chain.config_schema())  # Config schema accepted by runnable
print(chain.get_config_jsonschema())  # JSON schema of config
print(chain.get_graph())  # Graph view of runnable pipeline
print(chain.get_prompts())  # Prompts used (empty here)


RunnableSequence
<class 'langchain_core.runnables.base.fake_llm_input'>
{'title': 'fake_llm_input', 'type': 'string'}
<class 'langchain_core.output_parsers.json.JsonOutputParserOutput'>
{'title': 'JsonOutputParserOutput'}
<class 'langchain_core.utils.pydantic.RunnableSequenceConfig'>
{'properties': {}, 'title': 'RunnableSequenceConfig', 'type': 'object'}
Graph(nodes={'1edbd2b8534347ee89d431cdd581a89a': Node(id='1edbd2b8534347ee89d431cdd581a89a', name='fake_llm_input', data=<class 'langchain_core.runnables.base.fake_llm_input'>, metadata=None), '240df15bce3e46f1ac3e471f14ccc2d2': Node(id='240df15bce3e46f1ac3e471f14ccc2d2', name='fake_llm', data=RunnableLambda(fake_llm), metadata=None), '22d5555f8cd14698a93b4c614a04a3b7': Node(id='22d5555f8cd14698a93b4c614a04a3b7', name='JsonOutputParser', data=JsonOutputParser(pydantic_object=<class '__main__.MovieInfo'>), metadata=None), '964108cc46a448f5b625c5b702d4f5ef': Node(id='964108cc46a448f5b625c5b702d4f5ef', name='JsonOutputParserOutput', data=

In [8]:
# =============================
# 🔹 Runnable Composition
# =============================

chain2 = llm_runnable.__or__(parser)  # Same as |
chain3 = parser.__ror__(llm_runnable)  # Reverse pipe
chain4 = llm_runnable.pipe(parser)  # Explicit pipe method

In [9]:
# =============================
# 🔹 Output Dict Utilities
# =============================

picked = chain.pick(["title"])  # Picks only selected output keys
assigned = chain.assign(extra=lambda _:"Marvel")  # Adds extra field to output

In [10]:
# =============================
# 🔹 Invocation Methods
# =============================

print(chain.invoke("Tell me about Iron Man"))  # Sync call
print(chain.ainvoke("Tell me about Iron Man"))  # Async call (returns coroutine)

ValidationError: 1 validation error for Generation
text
  Input should be a valid string [type=string_type, input_value={'title': 'Iron Man', 'ye...irector': 'Jon Favreau'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [11]:
# =============================
# 🔹 Batch Execution
# =============================

inputs = ["Movie 1", "Movie 2"]

print(chain.batch(inputs))  # Parallel sync batch
print(chain.batch_as_completed(inputs))  # Batch results as they complete
print(chain.abatch(inputs))  # Async batch
print(chain.abatch_as_completed(inputs))  # Async batch completed

ValidationError: 1 validation error for Generation
text
  Input should be a valid string [type=string_type, input_value={'title': 'Iron Man', 'ye...irector': 'Jon Favreau'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [12]:
# =============================
# 🔹 Streaming
# =============================

print(chain.stream("Movie info"))  # Streaming output
print(chain.astream("Movie info"))  # Async streaming
print(chain.astream_log("Movie info"))  # Stream logs
print(chain.astream_events("Movie info"))  # Stream lifecycle events

<generator object RunnableSequence.stream at 0x00000296DB592020>
<async_generator object RunnableSequence.astream at 0x00000296DBC4D940>
<async_generator object Runnable.astream_log at 0x00000296D95B6880>
<async_generator object Runnable.astream_events at 0x00000296DBC54EE0>


In [13]:
# =============================
# 🔹 Transform APIs
# =============================

print(chain.transform("Movie info"))  # Transform input → output
print(chain.atransform("Movie info"))  # Async transform

<generator object RunnableSequence.transform at 0x00000296DB529000>
<async_generator object RunnableSequence.atransform at 0x00000296DB529000>


In [14]:
# =============================
# 🔹 Config & Binding
# =============================

print(chain.bind(debug=True))  # Bind static args
print(chain.with_config(tags=["exam"]))  # Attach config
print(chain.with_types(input_type=str, output_type=MovieInfo))  # Force types

bound=RunnableLambda(fake_llm)
| JsonOutputParser(pydantic_object=<class '__main__.MovieInfo'>) kwargs={'debug': True} config={} config_factories=[]
bound=RunnableLambda(fake_llm)
| JsonOutputParser(pydantic_object=<class '__main__.MovieInfo'>) kwargs={} config={'tags': ['exam']} config_factories=[]
bound=RunnableLambda(fake_llm)
| JsonOutputParser(pydantic_object=<class '__main__.MovieInfo'>) kwargs={} config={} config_factories=[] custom_input_type=<class 'str'> custom_output_type=<class '__main__.MovieInfo'>


In [15]:
# =============================
# 🔹 Listeners & Retry
# =============================

print(chain.with_listeners())  # Attach lifecycle listeners
print(chain.with_alisteners())  # Async listeners
print(chain.with_retry())  # Retry on failure

bound=RunnableLambda(fake_llm)
| JsonOutputParser(pydantic_object=<class '__main__.MovieInfo'>) kwargs={} config={} config_factories=[<function Runnable.with_listeners.<locals>.<lambda> at 0x00000296DBBDBE20>]
bound=RunnableLambda(fake_llm)
| JsonOutputParser(pydantic_object=<class '__main__.MovieInfo'>) kwargs={} config={} config_factories=[<function Runnable.with_alisteners.<locals>.<lambda> at 0x00000296DBBDBE20>]
bound=RunnableLambda(fake_llm)
| JsonOutputParser(pydantic_object=<class '__main__.MovieInfo'>) kwargs={} config={} config_factories=[]


In [16]:
# =============================
# 🔹 Mapping & Fallback
# =============================

print(chain.map())  # Map list inputs to list outputs
print(chain.with_fallbacks([llm_runnable]))  # Add fallback runnable

bound=RunnableLambda(fake_llm)
| JsonOutputParser(pydantic_object=<class '__main__.MovieInfo'>)
runnable=RunnableLambda(fake_llm)
| JsonOutputParser(pydantic_object=<class '__main__.MovieInfo'>) fallbacks=[RunnableLambda(fake_llm)]


In [17]:
# =============================
# 🔹 Tool Conversion
# =============================

tool = chain.as_tool()  # Convert runnable to LangChain tool

C:\Users\argroy\AppData\Local\Temp\ipykernel_39060\1569694374.py:5: LangChainBetaWarning: This API is in beta and may change in the future.
  tool = chain.as_tool()  # Convert runnable to LangChain tool


In [18]:
# =============================
# 🔹 Serialization
# =============================

print(chain.is_lc_serializable())  # Is serializable?
print(chain.get_lc_namespace())  # LangChain namespace
print(chain.lc_id())  # Unique serialization ID
print(chain.to_json())  # Serialize runnable
print(chain.dict())  # Dict representation

True
['langchain', 'schema', 'runnable']
['langchain', 'schema', 'runnable', 'RunnableSequence']
{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'schema', 'runnable', 'RunnableSequence'], 'kwargs': {'first': RunnableLambda(fake_llm), 'last': JsonOutputParser(pydantic_object=<class '__main__.MovieInfo'>)}, 'name': 'RunnableSequence'}
{'name': None, 'first': RunnableLambda(fake_llm), 'middle': [], 'last': {'name': None, 'diff': False, 'pydantic_object': <class '__main__.MovieInfo'>}}


C:\Users\argroy\AppData\Local\Temp\ipykernel_39060\793061935.py:9: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(chain.dict())  # Dict representation


In [19]:
# =============================
# 🔹 Output Parser Methods
# =============================

raw_text = '{"title":"Iron Man","year":2008,"director":"Jon Favreau"}'
print(parser.parse(raw_text))  # Parse single output
print(parser.parse_result([raw_text]))  # Parse generation list
print(parser.get_format_instructions())  # JSON format instructions
print(parser.parse_with_prompt(raw_text, "Movie prompt"))  # Parse with context
print(parser.aparse(raw_text))  # Async parse
print(parser.aparse_result([raw_text]))  # Async parse results


{'title': 'Iron Man', 'year': 2008, 'director': 'Jon Favreau'}


AttributeError: 'str' object has no attribute 'text'